# Занятие 9, пара 2. Решение без нейросетей


Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai


**Как проходить тетрадь.** Основные примеры работают без ответов «Ваш ход».
        Если пропускаете упражнение, пропустите также его `check(...)`. Продолжайте следующий обычный раздел.
        Сохраняйте свои попытки в отдельной копии. Проверки помогают разобраться, но не выставляют баллы.
        Интернет, Git и регистрация для основной работы не нужны.


## Сначала простой ответ, с которым можно сравнивать

Перед нейросетью нужен **baseline** — простое воспроизводимое решение. Оно показывает,
что даёт более сложный метод и действительно ли задача вообще устроена так, как мы думаем.
Сегодня распознаем реальные рукописные цифры, выберем признаки на validation и только затем посмотрим test.

После пары сможете собрать цепочку изображение → признаки → обученные эталоны классов → ответ,
прочитать матрицу ошибок и объяснить, где метод ошибается. Основной путь — NumPy/Matplotlib на CPU.
OpenCV и scikit-learn не обязательны. В конце есть отдельный переход к библиотечному Pipeline.
Все переменные создаются здесь, запуск 09.1 предварительно не нужен.


### Подключим учебные файлы

        Следующая ячейка — готовая служебная настройка. Она ищет `course_support` в рабочей папке
        или выше. Её не надо переписывать по памяти. `DATA_DIR` указывает на учебные данные,
        `WORKSPACE_DIR` — на корень вашей копии, `check` запускает проверку упражнения.
        Если поддержка не найдена, перенесите весь комплект, а не только notebook.
        В этой тетради создаются все необходимые переменные: другую тетрадь запускать не нужно.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check, summary

PROJECT_DIR


### 1. Реальные цифры, но уже подготовленные изображения

Набор UCI **Optical Recognition of Handwritten Digits**, E. Alpaydin и C. Kaynak, CC BY 4.0.
[Страница источника](https://archive.ics.uci.edu/dataset/80/optical+recognition+of+handwritten+digits).
Используем исходный вариант 32×32: бинарные пиксели, штрих=1, фон=0. Они уже выделены и центрированы.
Это проще, чем распознать произвольное фото документа: наша оценка относится к этой подготовленной задаче.

В комплекте три исходные части: train=1934, validation=946, test=1797. По описанию авторов
итоговый test содержит почерки других людей. Validation не объявляется независимой по людям от train.
Индивидуальные writer_id в этом формате не даны: проверяем происхождение по документации,
не придумываем недостающие ID. В своём проекте сохраняем их заранее, как обсуждали в 09.1.
Подробности, оригинальное описание и преобразования — `course_support/data/digits/SOURCE.md`.

Сначала читаем только train/validation. Test используем после выбора метода.


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

dataset = np.load(DATA_DIR / 'digits' / 'optdigits32.npz')
train_images = dataset['train_images'].astype(float)
train_labels = dataset['train_labels']
validation_images = dataset['validation_images'].astype(float)
validation_labels = dataset['validation_labels']
print(train_images.shape, validation_images.shape)
print('Диапазон пикселей:', train_images.min(), train_images.max())
fig, axes = plt.subplots(2, 5, figsize=(13, 6))
for class_id, ax in enumerate(axes.ravel()):
    index = int(np.flatnonzero(train_labels == class_id)[0])
    ax.imshow(train_images[index], cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    ax.set_title(f'Класс {class_id}')
    ax.axis('off')
plt.show()


### 2. Самый простой baseline вообще не смотрит на картинку

Всегда отвечать самым частым классом — слабый, но полезный ориентир.
`np.unique(..., return_counts=True)` возвращает метки и их количества; `argmax` выбирает наибольшее.
Частый класс определяем **по train**. Validation/test не должны подсказать, что предсказывать.


In [ ]:
classes, counts = np.unique(train_labels, return_counts=True)
majority_class = int(classes[np.argmax(counts)])
majority_validation = np.mean(validation_labels == majority_class)
print('Частый класс train:', majority_class)
print('Accuracy этого ответа на validation:', majority_validation)


### 3. Признак — что мы оставляем от изображения

Полная картинка содержит 1024 значения. Начнём с более короткого описания: разобьём её на клетки 4×4
и для каждой посчитаем среднюю яркость. В бинарной картинке это доля штриха внутри клетки.
Получим 8×8=64 числа, которые ещё сохраняют **место** штрихов.
Одна средняя по всей картинке этого не умеет: расположение потеряется.

Сначала одна клетка и одно число. Возьмём `image[4:8,12:16]`: четыре строки и четыре столбца на границе штриха. В ней есть и фон, и штрих; mean показывает долю штриха. На целой цифре отметим эту клетку рамкой.


In [ ]:
sample = train_images[0]
first_cell = sample[4:8, 12:16]
print(first_cell)
print('Доля штриха в клетке:', first_cell.mean())
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(sample, cmap='gray', vmin=0, vmax=1)
axes[0].add_patch(Rectangle((11.5,3.5),4,4,fill=False,edgecolor='lime',linewidth=2))
axes[0].set_title('Целая цифра и выбранная клетка')
axes[1].imshow(first_cell, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
axes[1].set_title('Клетка 4×4: 6 из 16 пикселей — штрих')
plt.show()


В выбранной клетке 6 единиц из 16, поэтому среднее равно 0.375. Теперь повторим для всех клеток слева направо, сверху вниз. `range(0,32,4)` даёт начала клеток 0,4,…,28.
Порядок — часть определения признака: на новом изображении он должен быть тем же.
Средние не обучаются по набору, но модель, которая будет использовать их, обучается.


In [ ]:
def block_features(image):
    values = []
    for y in range(0, 32, 4):
        for x in range(0, 32, 4):
            cell = image[y:y+4, x:x+4]
            values.append(cell.mean())
    return np.array(values)

sample_blocks = block_features(sample)
print('Длина признака:', len(sample_blocks))
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(sample_blocks.reshape(8, 8), cmap='gray', vmin=0, vmax=1, interpolation='nearest')
ax.set_title('Признак как карта заполнения клеток')
plt.show()


### ✏️ Ваш ход — Картинка видна, а вход другой

Наша модель ожидает штрих=1, фон=0. Новый сканер даёт uint8: чёрный штрих=0, белый фон=255.
Без исправления форму можно увидеть глазами, но смысл всех средних поменяется.
Напишите `prepare_scan(scan)`: линейно приведите 0…255 к нужному диапазону и полярности.
Проверьте на `[0, 127.5, 255]`. Почему преобразование надо применять одинаково при проверке и реальном использовании?
Считаем, что скан уже имеет размер 32×32; автоматическое выделение цифры здесь не решаем.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('09.2.1', prepare_scan)


**Мой вывод:** …


### 4. Другой признак: направления перепадов

Средние описали заполнение клеток. Можно описать **направления границ штрихов**.
Это основная идея HOG — Histogram of Oriented Gradients. Здесь построим упрощённое клеточное описание,
а не полную библиотечную реализацию HOG с перекрывающимися блоками и интерполяцией голосов.

`np.gradient(image)` вычисляет приближённые производные сначала по строкам (gy), затем по столбцам (gx).
`np.hypot(gx,gy)` — длина вектора градиента. `np.arctan2(gy,gx)` — его угол в радианах.
Операция `% np.pi` объединяет противоположные направления: нас интересует ориентация границы,
а не то, с какой её стороны светлее. Сам градиент направлен поперёк границы, а не вдоль неё.


In [ ]:
gy, gx = np.gradient(sample)
magnitude = np.hypot(gx, gy)
angles = np.arctan2(gy, gx) % np.pi
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(sample, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Штрих')
axes[1].imshow(magnitude, cmap='magma')
axes[1].set_title('Сила перепада на границе')
plt.show()


Теперь каждый пиксель голосует за интервал своего угла. Сильная граница даёт больший голос.
В `np.histogram(..., weights=...)` высота столбца — сумма весов, а не число точек.
Сначала три вымышленных градиента: два горизонтальных силы 1 и 3 и один вертикальный силы 2.
Ожидаем накопленные голоса 4 и 2 в соответствующих интервалах.


In [ ]:
tiny_angles = [0, 0, np.pi/2]
tiny_weights = [1, 3, 2]
tiny_edges = [0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi]
tiny_hist, edges = np.histogram(tiny_angles, bins=tiny_edges, weights=tiny_weights)
print('Взвешенная гистограмма:', tiny_hist)


Для цифры используем 4×4 клетки размера 8×8, в каждой восемь интервалов направления.
`np.linspace(0,np.pi,9)` создаёт девять границ, то есть восемь интервалов.
Нормируем гистограмму клетки на её длину: общий контраст меньше влияет на описание.
Малое число в знаменателе защищает пустую клетку от деления на ноль.
Итог 16×8=128 признаков. Сохраняем положение клеток — иначе 6 и 9 могут стать слишком похожи.

`ravel()` разворачивает клетку для histogram; `extend` добавляет восемь чисел в общий список. `np.linalg.norm` считает длину вектора как корень суммы квадратов.


In [ ]:
def direction_features(image):
    gy, gx = np.gradient(image.astype(float))
    magnitude = np.hypot(gx, gy)
    angles = np.arctan2(gy, gx) % np.pi
    edges = np.linspace(0, np.pi, 9)
    values = []
    for y in range(0, 32, 8):
        for x in range(0, 32, 8):
            cell_angles = angles[y:y+8, x:x+8].ravel()
            cell_weights = magnitude[y:y+8, x:x+8].ravel()
            histogram, _ = np.histogram(cell_angles, bins=edges, weights=cell_weights)
            length = np.linalg.norm(histogram)
            histogram = histogram / (length + 1e-8)
            values.extend(histogram)
    return np.array(values)

print('Длина второго признака:', len(direction_features(sample)))


`ravel()` только разворачивает клетку в список значений для histogram; `extend` добавляет все её
восемь чисел в общий список. Код внутри функции повторяет разобранные шаги.
Полный HOG использует более аккуратную нормировку и распределение голосов:
[Dalal & Triggs, 2005](https://lear.inrialpes.fr/people/triggs/pubs/Dalal-cvpr05.pdf).
Наш вариант полезен для понимания; его результат нельзя подписывать «стандартный HOG» без оговорки.

### 5. Что именно будем обучать

Пусть каждый класс описывается **средним вектором признаков его обучающих изображений** — центром класса.
Новая цифра получает класс ближайшего центра. Это классический nearest-centroid classifier.
Он проще множества других методов и не требует обучения нейросети.
Центр — усреднённое описание, не обязательно признак реально существовавшей цифры.

Сначала извлечём признаки всех train-изображений, затем посчитаем центр только нулей.


In [ ]:
train_blocks = np.array([block_features(image) for image in train_images])
zero_center = train_blocks[train_labels == 0].mean(axis=0)
print('Таблица признаков train:', train_blocks.shape)
print('Центр класса 0:', zero_center.shape)
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(zero_center.reshape(8, 8), cmap='gray', vmin=0, vmax=1)
ax.set_title('Среднее расположение штрихов класса 0 в train')
plt.show()


Теперь центры всех классов. `mean(axis=0)` усредняет изображения, оставляя отдельное значение
для каждой координаты признака. Validation сюда не добавляем.
Для ответа считаем сумму квадратов разностей до каждого центра; квадратный корень не нужен,
потому что он не изменит, какое расстояние меньше.


In [ ]:
def fit_centers(features, labels):
    class_ids = np.unique(labels)
    centers = []
    for class_id in class_ids:
        centers.append(features[labels == class_id].mean(axis=0))
    return class_ids, np.array(centers)

def predict_centers(features, class_ids, centers):
    predictions = []
    for vector in features:
        differences = centers - vector
        squared_distances = np.sum(differences**2, axis=1)
        nearest = np.argmin(squared_distances)
        predictions.append(class_ids[nearest])
    return np.array(predictions)

class_ids, centers = fit_centers(train_blocks, train_labels)
first_validation = block_features(validation_images[0])
first_prediction = predict_centers(np.array([first_validation]), class_ids, centers)
print('Первый ответ:', first_prediction[0], 'метка:', validation_labels[0])


### 6. Выбираем по validation, а не по знакомому train

Сравним два заранее названных способа описания: заполнение клеток и направления границ.
Классификатор и части данных одинаковые. Сохраняем признаки и центры, чтобы затем не пересчитать
их по другому правилу. Accuracy — доля совпавших меток. В этом учебном наборе классы близки по численности;
при сильном дисбалансе одной accuracy обычно недостаточно.


In [ ]:
feature_functions = {'blocks': block_features, 'directions': direction_features}
trained = {}
validation_scores = {}
for name, feature_function in feature_functions.items():
    x_train = np.array([feature_function(image) for image in train_images])
    x_validation = np.array([feature_function(image) for image in validation_images])
    class_ids, centers = fit_centers(x_train, train_labels)
    predicted = predict_centers(x_validation, class_ids, centers)
    validation_scores[name] = float(np.mean(predicted == validation_labels))
    trained[name] = {'classes':class_ids, 'centers':centers,
                     'x_train':x_train, 'x_validation':x_validation}
print('Постоянный ответ:', majority_validation)
print('Признаки + центры:', validation_scores)


### ✏️ Ваш ход — Самая точная модель не укладывается в задачу

Для другого эксперимента получены записи с полями name, validation_accuracy, latency_ms.
Нужно выбрать наибольшую validation_accuracy **среди вариантов, укладывающихся в max_ms**.
Напишите `choose_feasible(rows, max_ms)`, возвращающую имя; при отсутствии допустимых вариантов — None.
При равенстве качества оставьте первый допустимый вариант. Test-оценки функции не передаются.

Пример: A — 0.91 и 80 мс, B — 0.89 и 12 мс, C — 0.87 и 5 мс. Ограничение 20 мс.
Объясните, почему здесь обоснованно выбрать не максимальную accuracy из всех строк.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('09.2.2', choose_feasible)


**Мой вывод:** …


### 7. Фиксируем выбор и только затем читаем test

Для основного опыта выбираем по accuracy validation; при равенстве остаётся первый вариант в словаре.
`max(словарь, key=словарь.get)` выбирает ключ с наибольшим значением.
Порогов, числа клеток и seed после просмотра test не подбираем.
Если по результату захотелось изменить метод, это новый эксперимент; прежний test уже не «невидимый».


In [ ]:
chosen_name = max(validation_scores, key=validation_scores.get)
chosen_model = trained[chosen_name]
chosen_feature = feature_functions[chosen_name]
print('Зафиксированный выбор:', chosen_name)

test_images = dataset['test_images'].astype(float)
test_labels = dataset['test_labels']
x_test = np.array([chosen_feature(image) for image in test_images])
test_predictions = predict_centers(x_test, chosen_model['classes'], chosen_model['centers'])
baseline_test = float(np.mean(test_labels == majority_class))
test_accuracy = float(np.mean(test_predictions == test_labels))
print('Test: постоянный ответ:', baseline_test)
print('Test: выбранный метод:', test_accuracy)


### 8. В каких классах ошибаемся?

**Матрица ошибок:** строка — истинный класс, столбец — предсказанный.
Диагональ содержит правильные ответы. Например, клетка [3,8] считает настоящие тройки,
которые назвали восьмёрками. Не меняем эти оси местами.
Сначала создаём нули, затем для каждой пары меток увеличиваем соответствующую клетку.


In [ ]:
confusion = np.zeros((10, 10), dtype=int)
for true_label, predicted_label in zip(test_labels, test_predictions):
    confusion[true_label, predicted_label] += 1
assert confusion.sum() == len(test_labels)
fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(confusion, cmap='Blues')
fig.colorbar(image, ax=ax, label='Количество примеров')
for row in range(10):
    for col in range(10):
        color = 'white' if confusion[row,col] > confusion.max()/2 else 'black'
        ax.text(col,row,str(confusion[row,col]),ha='center',va='center',color=color)
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xlabel('Предсказанный класс')
ax.set_ylabel('Истинный класс')
ax.set_title('Ошибки выбранного метода на фиксированном test')
plt.show()


Для класса 8: TP — [8,8], FN — остальные клетки строки 8, FP — остальные клетки столбца 8.
Recall делит TP на сумму строки: сколько настоящих восьмёрок нашли.
Precision делит TP на сумму столбца: сколько из ответов «8» действительно восьмёрки.
Если знаменатель нулевой, пишем None, а не объявляем идеальное качество.


In [ ]:
class_id = 8
tp = int(confusion[class_id,class_id])
actual_count = int(confusion[class_id].sum())
predicted_count = int(confusion[:,class_id].sum())
recall_eight = tp / actual_count if actual_count else None
precision_eight = tp / predicted_count if predicted_count else None
print('Класс 8: precision:', precision_eight, 'recall:', recall_eight)
recalls = []
for class_id in range(10):
    count = confusion[class_id].sum()
    if count > 0:
        recalls.append(confusion[class_id,class_id] / count)
print('Средняя полнота по представленным классам:', np.mean(recalls))


### ✏️ Ваш ход — Высокая accuracy скрыла целый класс

В другом наборе 95 объектов класса 0 и 5 класса 1. Модель всегда отвечает 0.
Составьте `blind_example` — матрицу ошибок 2×2. Accuracy будет 95%, но один класс не найден ни разу.
Напишите `blind_classes(matrix)`: множество индексов классов, которые присутствуют в истине,
но не имеют ни одного правильного ответа. Класс с пустой строкой не включаем.
Объясните, почему отсутствие примеров класса в test — другая проблема, а не доказательство хорошего качества.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('09.2.3', blind_example, blind_classes)


**Мой вывод:** …


### 9. Посмотрим на ошибки, а не только на число

`np.flatnonzero` вернёт индексы неверных ответов. Покажем первые несколько в исходном порядке;
это иллюстрации, не случайная репрезентативная выборка всех ошибок.
Ищем общий признак: недописанный штрих, похожая форма, смещение, необычный почерк.
Одна правдоподобная история по одной картинке ещё не доказывает причину всех ошибок.


In [ ]:
error_indices = np.flatnonzero(test_predictions != test_labels)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax in axes.ravel():
    ax.axis('off')
for ax, index in zip(axes.ravel(), error_indices[:8]):
    ax.imshow(test_images[index], cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    ax.set_title(f'Истина {test_labels[index]}, ответ {test_predictions[index]}')
fig.suptitle('Ошибки на настоящих рукописных цифрах')
plt.show()
print('Всего ошибок:', len(error_indices))


### 10. Сохраним эксперимент

Сохраняем выбранный способ, показатели и центры. Список признаков и исходное разбиение должны остаться теми же,
когда позже сравним с CNN. Одних весов/центров без подготовки входа недостаточно для воспроизведения.
В своём проекте код чтения данных и функций признаков также должен входить в выдачу.

`np.savetxt` сохраняет матрицу текстом: `fmt="%d"` — целые числа, `delimiter=","` — разделитель CSV. JSON хранит описание опыта, NPZ — численные центры.


In [ ]:
output_dir = WORKSPACE_DIR / 'outputs' / 'classical_digits'
output_dir.mkdir(parents=True, exist_ok=True)
result = {'feature':chosen_name, 'validation':validation_scores,
          'baseline_test_accuracy':baseline_test, 'test_accuracy':test_accuracy,
          'split':'Original UCI tra/cv/windep; see data/digits/SOURCE.md'}
(output_dir / 'result.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
np.savez_compressed(output_dir / 'centers.npz',
                    classes=chosen_model['classes'], centers=chosen_model['centers'])
np.savetxt(output_dir / 'confusion.csv', confusion, fmt='%d', delimiter=',')
print(output_dir)


## ЛР5 — тот же ход на данных проекта

Задание в `docs/ЛР5.md` в корне вашей копии, прежние 15 баллов.
Постройте простой baseline, извлеките подходящие признаки, обучите классический классификатор без утечки,
выберите настройки на train/validation, зафиксируйте их и сравните на test. Разберите ошибки.
Нет обязательного порога точности и обязательного превосходства нейросети в будущем.
Nearest-centroid допустим, если выбор признаков и его ограничения объяснены. Библиотека не оценивается вместо метода.

### Дополнительно: знакомый sklearn Pipeline

Если scikit-learn доступен, можно сравнить на **validation** заранее заданный линейный SVM.
`StandardScaler` обучается по train, `Pipeline` объединяет преобразование и классификатор,
`fit` обучает оба шага, `predict` применяет их к новым примерам. C=1 фиксирован заранее.
Основной тестовый результат выше от этой ветки не меняется; не подбирайте параметры по уже увиденному test.
Если хотите новый честный выбор между методами — запланируйте его до следующего отложенного теста.
Основной маршрут уже завершён; переключатель False оставляет расширение выключенным.


In [ ]:
RUN_SKLEARN = False
if RUN_SKLEARN:
    try:
        from sklearn.pipeline import make_pipeline
        from sklearn.preprocessing import StandardScaler
        from sklearn.svm import LinearSVC
    except ImportError:
        print('scikit-learn недоступен; основной NumPy-эксперимент уже выполнен.')
    else:
        pipeline = make_pipeline(StandardScaler(), LinearSVC(C=1, max_iter=10000, random_state=42))
        pipeline.fit(chosen_model['x_train'], train_labels)
        optional_validation = pipeline.predict(chosen_model['x_validation'])
        print('Дополнительный фиксированный SVM, validation:',
              np.mean(optional_validation == validation_labels))


[Официальная памятка sklearn об утечках и Pipeline](https://scikit-learn.org/stable/common_pitfalls.html).

**Резерв на 10–15 минут:** возьмите одну validation-цифру и сдвиньте на 2 пикселя в пустом поле,
не оборачивая края на другую сторону. Сравните признаки и ответ. Почему одинаковая цифра может оказаться
далеко от центра класса? Это отдельная проверка устойчивости на validation, не новая настройка по test.

Главное после пары: понятно, что было обучено, что выбиралось, на каких данных получено итоговое число
и какие ошибки оно скрывает. Следующий блок начнёт строить нейросети; сегодняшнее решение останется точкой сравнения.
